# BinX Week 07 — Day 2: Building CNNs & Transfer Learning

| Field | Value |
|:------|:------|
| **Phase** | Phase 3 — Deep Learning & Applied Project |
| **Sprint** | Sprint 2 (Weeks 6–9) |
| **Day** | Day 2 of 5 |
| **Dataset** | Skin Lesion Images — Binary (Benign vs Malignant) |
| **Image Size** | 224 × 224 × 3 (RGB JPEG) |
| **Train** | 11,879 images (6,289 Benign + 5,590 Malignant) |
| **Test** | 2,000 images (1,000 Benign + 1,000 Malignant) |
| **Notebook** | `BinX_Week_07/Day2/Building_CNNs.ipynb` |

---

## Day 2 Learning Objectives

1. Build a full CNN with convolution, pooling, and dense layers from scratch
2. Apply data augmentation to reduce overfitting on image data
3. Use transfer learning with a pre-trained model (MobileNetV2) to obtain strong results from limited data
4. Compare all three approaches fairly and document findings

---

## Connection to Day 1

Day 1 introduced convolution as a mathematical operation, demonstrated edge detection with hand-defined filters, and established the principle that **architecture must match data type**. Today we build on that foundation by:

- Moving from hand-defined filters to **learned filters** (trainable CNN)
- Adding **pooling** for spatial downsampling
- Constructing a **complete trainable network** with Conv → Pool → Dense layers
- Exploring **data augmentation** and **transfer learning** as practical techniques

### Important Architecture Note

Day 1 established that our project's **Heart Disease dataset is tabular** (918 patients × 14 columns) and therefore **CNNs are not the correct architecture for the core model**. The core model for the project continues to be the **dense network from Week 6**, enhanced through Sprint 2.

This Day 2 CNN laboratory uses **real medical images** (skin lesion classification) — a task where CNNs are the correct architecture. This is the required computer-vision educational experiment, implemented separately from the project's core tabular model. The medical imaging context is also relevant to the broader healthcare domain of the Cardiac Patient Monitoring System project.

---
## Architecture Selection — Why CNN Here, Dense for the Project

### Project Data Type: **TABULAR**

Our project uses the Heart Disease dataset: 918 independent patient records, each described by 13 numeric features + 1 categorical feature. There is:
- **No spatial grid** (pixels in 2D) → CNN is not appropriate
- **No temporal sequence** (words, timestamps) → RNN/Transformer is not appropriate
- **Independent feature vectors** → Dense network is the correct choice

### Why We Use CNN Today

The Week 7 curriculum requires hands-on experience with CNNs, data augmentation, and transfer learning. Since these techniques apply specifically to **image data**, we use the **skin lesion image dataset** — a medically relevant binary classification task (Benign vs Malignant).

| Component | Dataset | Architecture |
|:----------|:--------|:-------------|
| **Core Project** | Heart Disease (tabular) | Dense Network (Week 6) |
| **Day 2 Educational Lab** | Skin Lesion Images (224×224 RGB) | CNN + Transfer Learning |

This separation is intentional and correct: **we do not force a CNN onto tabular data.**

In [ ]:
# ─── Core Libraries ───
import numpy as np
import matplotlib.pyplot as plt
import os
import time
import warnings
warnings.filterwarnings('ignore')

# ─── TensorFlow / Keras ───
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess

# ─── Scikit-learn ───
from sklearn.metrics import classification_report, confusion_matrix

# ─── Environment Detection ───
print("=" * 60)
print("ENVIRONMENT")
print("=" * 60)
print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version:      {keras.__version__}")
print(f"NumPy version:      {np.__version__}")
gpu_devices = tf.config.list_physical_devices('GPU')
cpu_devices = tf.config.list_physical_devices('CPU')
print(f"GPU devices:        {len(gpu_devices)}")
print(f"CPU devices:        {len(cpu_devices)}")
print(f"Execution device:   {'GPU' if gpu_devices else 'CPU (all experiments run on CPU)'}")
print("=" * 60)

# ─── Reproducibility ───
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print(f"\nRandom seed set to {SEED} for reproducibility.")

# ─── Plot Defaults ───
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 10

print("\n[OK] All libraries loaded successfully.")

---
## Dataset & Problem Definition

### Skin Lesion Image Dataset

We use a binary image classification dataset of skin lesions:

| Property | Value |
|:---------|:------|
| **Task** | Binary classification: Benign vs Malignant |
| **Training samples** | 11,879 (6,289 Benign + 5,590 Malignant) |
| **Test samples** | 2,000 (1,000 Benign + 1,000 Malignant) |
| **Image size** | 224 × 224 pixels, 3 channels (RGB) |
| **Format** | JPEG |
| **Class balance** | Slightly imbalanced (53% Benign / 47% Malignant in training) |

### Why This Dataset?

1. **Medically relevant** — connects to the healthcare domain of our cardiac project
2. **Real images** (not synthetic) — teaches practical data loading and preprocessing
3. **224×224 RGB** — matches MobileNetV2's native input size for transfer learning
4. **Binary classification** — clean, interpretable results
5. **Moderate size** (~12K train images) — trainable on CPU in reasonable time

In [ ]:
# ─── Configuration ───
DATA_DIR = '../../Data/images-dataset'
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
TEST_DIR = os.path.join(DATA_DIR, 'test')
IMG_SIZE = 128        # Resize for CNN from scratch (smaller = faster on CPU)
IMG_SIZE_TL = 224     # MobileNetV2 native input size
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# ─── Count Images ───
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
for split_name, split_dir in [('Train', TRAIN_DIR), ('Test', TEST_DIR)]:
    total = 0
    for cls in sorted(os.listdir(split_dir)):
        cls_path = os.path.join(split_dir, cls)
        if os.path.isdir(cls_path):
            n = len([f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
            print(f"  {split_name}/{cls}: {n} images")
            total += n
    print(f"  {split_name} total: {total} images")
print("=" * 60)

# ─── Load Full Training Set ───
full_train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels='inferred',
    label_mode='binary',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
)

CLASS_NAMES = full_train_ds.class_names
print(f"\nClass names: {CLASS_NAMES}")
print(f"Number of classes: {len(CLASS_NAMES)}")

# ─── Train / Validation Split (80/20) ───
val_size = int(0.2 * len(full_train_ds))
train_ds = full_train_ds.skip(val_size)
val_ds = full_train_ds.take(val_size)

# Prefetch for performance
train_ds = train_ds.cache().shuffle(1000, seed=SEED).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

print(f"\nTrain batches: {len(train_ds)} (~{len(train_ds) * BATCH_SIZE} images)")
print(f"Val batches:   {len(val_ds)} (~{len(val_ds) * BATCH_SIZE} images)")

# ─── Load Test Set (at CNN input size) ───
test_ds_cnn = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    labels='inferred',
    label_mode='binary',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False,
)
test_ds_cnn = test_ds_cnn.cache().prefetch(AUTOTUNE)

# ─── Load Test Set (at MobileNetV2 input size) ───
test_ds_tl = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    labels='inferred',
    label_mode='binary',
    image_size=(IMG_SIZE_TL, IMG_SIZE_TL),
    batch_size=BATCH_SIZE,
    shuffle=False,
)
test_ds_tl = test_ds_tl.cache().prefetch(AUTOTUNE)

print(f"\nTest batches (CNN): {len(test_ds_cnn)}")
print(f"Test batches (TL):  {len(test_ds_tl)}")

In [ ]:
# ─── Visualize Sample Images ───
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle('Sample Images from Dataset', fontsize=14, fontweight='bold')

for images, labels in train_ds.take(1):
    for i in range(min(10, len(images))):
        ax = axes[i // 5, i % 5]
        img = images[i].numpy().astype('uint8')
        label = CLASS_NAMES[int(labels[i].numpy().item())]
        ax.imshow(img)
        ax.set_title(label, fontsize=10)
        ax.axis('off')

plt.tight_layout()
plt.show()
print(f"\n[OK] Visualized sample images at {IMG_SIZE}x{IMG_SIZE} resolution.")

---
## Experiment 1 — CNN From Scratch

We build a complete CNN containing all the concepts required by the curriculum:

```
Input (128×128×3)
↓
Conv2D (32 filters, 3×3, ReLU) + MaxPooling2D (2×2)
↓
Conv2D (64 filters, 3×3, ReLU) + MaxPooling2D (2×2)
↓
Conv2D (128 filters, 3×3, ReLU) + MaxPooling2D (2×2)
↓
Flatten
↓
Dense (128, ReLU) + Dropout (0.5)
↓
Dense (1, Sigmoid)
```

### Architecture Rationale
- **Three convolutional blocks** progressively extract features: edges → textures → patterns
- **MaxPooling2D** reduces spatial dimensions, adds translation invariance, and reduces computation
- **Flatten** converts 2D feature maps to 1D vector for dense classification
- **Sigmoid output** produces probability for binary classification
- Images resized to 128×128 to keep training fast on CPU

In [ ]:
# ─── Data Normalization Helper ───
normalization_layer = layers.Rescaling(1.0 / 255)

# Apply normalization to train/val/test (for CNN from scratch)
train_ds_norm = train_ds.map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=AUTOTUNE)
val_ds_norm = val_ds.map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=AUTOTUNE)
test_ds_norm = test_ds_cnn.map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=AUTOTUNE)

# ─── Build Baseline CNN ───
def build_baseline_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 3)):
    model = models.Sequential([
        # Block 1: Conv → ReLU → MaxPool
        layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                      input_shape=input_shape, name='conv1'),
        layers.MaxPooling2D((2, 2), name='pool1'),
        
        # Block 2: Conv → ReLU → MaxPool
        layers.Conv2D(64, (3, 3), activation='relu', padding='same', name='conv2'),
        layers.MaxPooling2D((2, 2), name='pool2'),
        
        # Block 3: Conv → ReLU → MaxPool
        layers.Conv2D(128, (3, 3), activation='relu', padding='same', name='conv3'),
        layers.MaxPooling2D((2, 2), name='pool3'),
        
        # Classification head
        layers.Flatten(name='flatten'),
        layers.Dense(128, activation='relu', name='dense1'),
        layers.Dropout(0.5, name='dropout'),
        layers.Dense(1, activation='sigmoid', name='output')
    ])
    return model

baseline_model = build_baseline_cnn()
baseline_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("=" * 60)
print("BASELINE CNN ARCHITECTURE")
print("=" * 60)
baseline_model.summary()
print(f"\nTotal parameters: {baseline_model.count_params():,}")

In [ ]:
# ─── Train Baseline CNN ───
print("Training Baseline CNN...")
print("=" * 60)

early_stop = callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
)

start_time = time.time()
baseline_history = baseline_model.fit(
    train_ds_norm,
    epochs=15,
    validation_data=val_ds_norm,
    callbacks=[early_stop],
    verbose=1
)
baseline_train_time = time.time() - start_time

print(f"\n{'=' * 60}")
print(f"Training completed in {baseline_train_time:.1f} seconds")
print(f"Epochs trained: {len(baseline_history.history['loss'])}")
print(f"{'=' * 60}")

In [ ]:
# ─── Evaluate Baseline CNN ───
baseline_train_acc = baseline_history.history['accuracy']
baseline_val_acc = baseline_history.history['val_accuracy']
baseline_train_loss = baseline_history.history['loss']
baseline_val_loss = baseline_history.history['val_loss']

baseline_best_val_acc = max(baseline_val_acc)
baseline_best_epoch = np.argmax(baseline_val_acc) + 1

baseline_test_loss, baseline_test_acc = baseline_model.evaluate(test_ds_norm, verbose=0)

print("=" * 60)
print("BASELINE CNN RESULTS")
print("=" * 60)
print(f"Training time:            {baseline_train_time:.1f}s")
print(f"Epochs trained:           {len(baseline_train_acc)}")
print(f"Best validation accuracy: {baseline_best_val_acc:.4f} (epoch {baseline_best_epoch})")
print(f"Test accuracy:            {baseline_test_acc:.4f}")
print(f"Test loss:                {baseline_test_loss:.4f}")
print("=" * 60)

# ─── Plot Training Curves ───
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Experiment 1: Baseline CNN — Training Curves', fontsize=14, fontweight='bold')

epochs_range = range(1, len(baseline_train_acc) + 1)
ax1.plot(epochs_range, baseline_train_acc, 'b-o', label='Training Accuracy', markersize=4)
ax1.plot(epochs_range, baseline_val_acc, 'r-o', label='Validation Accuracy', markersize=4)
ax1.axhline(y=baseline_test_acc, color='green', linestyle='--', alpha=0.7, label=f'Test Acc ({baseline_test_acc:.4f})')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy'); ax1.set_title('Accuracy'); ax1.legend(); ax1.set_ylim([0.4, 1.0])

ax2.plot(epochs_range, baseline_train_loss, 'b-o', label='Training Loss', markersize=4)
ax2.plot(epochs_range, baseline_val_loss, 'r-o', label='Validation Loss', markersize=4)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss'); ax2.set_title('Loss'); ax2.legend()

plt.tight_layout()
plt.show()

gap = baseline_train_acc[-1] - baseline_val_acc[-1]
print(f"\nTrain-Val accuracy gap (last epoch): {gap:.4f}")
if gap > 0.05:
    print("[WARN] Signs of overfitting detected (gap > 5%). Data augmentation may help.")
else:
    print("[OK] No strong overfitting detected (gap <= 5%).")

---
## Experiment 2 — CNN + Data Augmentation

Data augmentation artificially expands the training set by applying random transformations to images.

### Augmentation Transforms Applied

| Transform | What It Does | Why It Helps |
|:----------|:-------------|:-------------|
| **RandomFlip** | Flips image horizontally | Teaches invariance to left-right mirroring |
| **RandomRotation** | Rotates by small angle | Teaches invariance to slight tilts |
| **RandomZoom** | Zooms in/out slightly | Teaches scale invariance |

Augmentation is applied **only during training**. The validation and test sets remain unaugmented.

In [ ]:
# ─── Build Augmentation Pipeline + CNN ───
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal", name='random_flip'),
    layers.RandomRotation(0.15, name='random_rotation'),
    layers.RandomZoom(0.15, name='random_zoom'),
], name='data_augmentation')

# Using Functional API because data_augmentation is a nested Sequential model.
# In Sequential, a nested Sequential without input_shape prevents the parent
# from inferring shapes. Functional API traces through all layers correctly.
input_layer = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='input')
x = data_augmentation(input_layer)
x = layers.Rescaling(1.0 / 255, name='rescaling')(x)

# Same architecture as baseline
x = layers.Conv2D(32, (3, 3), activation='relu', padding='same', name='conv1')(x)
x = layers.MaxPooling2D((2, 2), name='pool1')(x)
x = layers.Conv2D(64, (3, 3), activation='relu', padding='same', name='conv2')(x)
x = layers.MaxPooling2D((2, 2), name='pool2')(x)
x = layers.Conv2D(128, (3, 3), activation='relu', padding='same', name='conv3')(x)
x = layers.MaxPooling2D((2, 2), name='pool3')(x)

x = layers.Flatten(name='flatten')(x)
x = layers.Dense(128, activation='relu', name='dense1')(x)
x = layers.Dropout(0.5, name='dropout')(x)
output = layers.Dense(1, activation='sigmoid', name='output')(x)

augmented_model = models.Model(inputs=input_layer, outputs=output, name='augmented_cnn')
augmented_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("=" * 60)
print("AUGMENTED CNN ARCHITECTURE")
print("=" * 60)
augmented_model.summary()

In [ ]:
# ─── Visualize Augmentation Effects ───
for images, labels in train_ds.take(1):
    sample_image = images[:1]
    break

fig, axes = plt.subplots(2, 5, figsize=(14, 5))
fig.suptitle('Data Augmentation: Original + 9 Augmented Versions', fontsize=14, fontweight='bold')

axes[0, 0].imshow(sample_image[0].numpy().astype('uint8'))
axes[0, 0].set_title('Original', fontweight='bold')
axes[0, 0].axis('off')

for i in range(1, 10):
    aug = data_augmentation(sample_image, training=True)
    row, col = divmod(i, 5)
    axes[row, col].imshow(aug[0].numpy().astype('uint8'))
    axes[row, col].set_title(f'Augmented #{i}')
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()
print("[OK] Each augmented version applies random flips, rotations, and zooms.")

In [ ]:
# ─── Train Augmented CNN ───
print("Training Augmented CNN...")
print("=" * 60)

early_stop_aug = callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
)

start_time = time.time()
augmented_history = augmented_model.fit(
    train_ds,
    epochs=15,
    validation_data=val_ds,
    callbacks=[early_stop_aug],
    verbose=1
)
augmented_train_time = time.time() - start_time

print(f"\n{'=' * 60}")
print(f"Training completed in {augmented_train_time:.1f} seconds")
print(f"Epochs trained: {len(augmented_history.history['loss'])}")
print(f"{'=' * 60}")

In [ ]:
# ─── Evaluate Augmented CNN ───
aug_train_acc = augmented_history.history['accuracy']
aug_val_acc = augmented_history.history['val_accuracy']
aug_train_loss = augmented_history.history['loss']
aug_val_loss = augmented_history.history['val_loss']

aug_best_val_acc = max(aug_val_acc)
aug_best_epoch = np.argmax(aug_val_acc) + 1

aug_test_loss, aug_test_acc = augmented_model.evaluate(test_ds_cnn, verbose=0)

print("=" * 60)
print("AUGMENTED CNN RESULTS")
print("=" * 60)
print(f"Training time:            {augmented_train_time:.1f}s")
print(f"Epochs trained:           {len(aug_train_acc)}")
print(f"Best validation accuracy: {aug_best_val_acc:.4f} (epoch {aug_best_epoch})")
print(f"Test accuracy:            {aug_test_acc:.4f}")
print(f"Test loss:                {aug_test_loss:.4f}")
print("=" * 60)

# ─── Plot Training Curves ───
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Experiment 2: CNN + Data Augmentation — Training Curves', fontsize=14, fontweight='bold')

epochs_range = range(1, len(aug_train_acc) + 1)
ax1.plot(epochs_range, aug_train_acc, 'b-o', label='Training Accuracy', markersize=4)
ax1.plot(epochs_range, aug_val_acc, 'r-o', label='Validation Accuracy', markersize=4)
ax1.axhline(y=aug_test_acc, color='green', linestyle='--', alpha=0.7, label=f'Test Acc ({aug_test_acc:.4f})')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy'); ax1.set_title('Accuracy'); ax1.legend(); ax1.set_ylim([0.4, 1.0])

ax2.plot(epochs_range, aug_train_loss, 'b-o', label='Training Loss', markersize=4)
ax2.plot(epochs_range, aug_val_loss, 'r-o', label='Validation Loss', markersize=4)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss'); ax2.set_title('Loss'); ax2.legend()

plt.tight_layout()
plt.show()

aug_gap = aug_train_acc[-1] - aug_val_acc[-1]
print(f"\nTrain-Val accuracy gap (last epoch): {aug_gap:.4f}")
print(f"Baseline gap was: {gap:.4f}")
if aug_gap < gap:
    print("[OK] Augmentation reduced the overfitting gap.")
else:
    print("[WARN] Augmentation did not reduce the overfitting gap this time.")

---
## Validation Curve Comparison — Baseline vs. Augmentation

In [ ]:
# ─── Side-by-Side Comparison ───
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Baseline vs. Augmented CNN — Validation Curves', fontsize=14, fontweight='bold')

max_epochs = max(len(baseline_val_acc), len(aug_val_acc))
baseline_val_padded = list(baseline_val_acc) + [None] * (max_epochs - len(baseline_val_acc))
aug_val_padded = list(aug_val_acc) + [None] * (max_epochs - len(aug_val_acc))
baseline_val_loss_padded = list(baseline_val_loss) + [None] * (max_epochs - len(baseline_val_loss))
aug_val_loss_padded = list(aug_val_loss) + [None] * (max_epochs - len(aug_val_loss))

epochs_range = range(1, max_epochs + 1)

axes[0].plot(epochs_range, baseline_val_padded, 'b-o', label='Baseline (no aug)', markersize=4)
axes[0].plot(epochs_range, aug_val_padded, 'r-o', label='With Augmentation', markersize=4)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Validation Accuracy'); axes[0].set_title('Validation Accuracy')
axes[0].legend(); axes[0].set_ylim([0.4, 1.0])

axes[1].plot(epochs_range, baseline_val_loss_padded, 'b-o', label='Baseline (no aug)', markersize=4)
axes[1].plot(epochs_range, aug_val_loss_padded, 'r-o', label='With Augmentation', markersize=4)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Validation Loss'); axes[1].set_title('Validation Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Baseline  — Best Val Acc: {baseline_best_val_acc:.4f}, Test Acc: {baseline_test_acc:.4f}")
print(f"Augmented — Best Val Acc: {aug_best_val_acc:.4f}, Test Acc: {aug_test_acc:.4f}")
diff_val = aug_best_val_acc - baseline_best_val_acc
diff_test = aug_test_acc - baseline_test_acc
print(f"\nAugmentation impact: Val Acc {diff_val:+.4f}, Test Acc {diff_test:+.4f}")

---
## Experiment 3 — Transfer Learning

Transfer learning leverages a model pre-trained on ImageNet and adapts it to our task.

### Approach: Frozen MobileNetV2 + New Classification Head

```
Pre-trained MobileNetV2 (frozen — weights NOT updated)
↓
GlobalAveragePooling2D
↓
Dense (128, ReLU) + Dropout (0.3)
↓
Dense (1, Sigmoid)
```

### Key Details
- **MobileNetV2**: Lightweight architecture pre-trained on 1.4M ImageNet images
- **Frozen backbone**: All pre-trained weights are frozen — only the new head is trained
- **Input size**: 224×224×3 — matches our raw image size exactly!
- **Preprocessing**: MobileNetV2 expects pixels in [-1, 1]

In [ ]:
# ─── Prepare Data for MobileNetV2 ───
# Apply MobileNetV2 preprocessing to train/val/test at 224×224
def apply_mobilenet_preprocess(image, label):
    image = tf.cast(image, tf.float32)
    image = mobilenet_preprocess(image)
    return image, label

# Reload train/val at 224×224 for transfer learning
train_ds_224 = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, labels='inferred', label_mode='binary',
    image_size=(IMG_SIZE_TL, IMG_SIZE_TL), batch_size=BATCH_SIZE,
    shuffle=True, seed=SEED
)
val_size_tl = int(0.2 * len(train_ds_224))
train_ds_tl_raw = train_ds_224.skip(val_size_tl)
val_ds_tl_raw = train_ds_224.take(val_size_tl)

train_ds_tl = train_ds_tl_raw.map(apply_mobilenet_preprocess, num_parallel_calls=AUTOTUNE).cache().shuffle(1000, seed=SEED).prefetch(AUTOTUNE)
val_ds_tl = val_ds_tl_raw.map(apply_mobilenet_preprocess, num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)

test_ds_tl_prep = test_ds_tl.map(apply_mobilenet_preprocess, num_parallel_calls=AUTOTUNE)

print(f"Train batches (224×224): {len(train_ds_tl)}")
print(f"Val batches (224×224):   {len(val_ds_tl)}")
print(f"Test batches (224×224):  {len(test_ds_tl_prep)}")

# ─── Build Transfer Learning Model ───
base_model = MobileNetV2(
    input_shape=(IMG_SIZE_TL, IMG_SIZE_TL, 3),
    include_top=False,
    weights='imagenet',
    pooling=None
)
base_model.trainable = False  # Freeze the backbone

transfer_model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(name='global_avg_pool'),
    layers.Dense(128, activation='relu', name='dense1'),
    layers.Dropout(0.3, name='dropout'),
    layers.Dense(1, activation='sigmoid', name='output')
])

transfer_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("\n" + "=" * 60)
print("TRANSFER LEARNING MODEL (MobileNetV2 — Frozen Backbone)")
print("=" * 60)
transfer_model.summary()

frozen_params = sum([tf.size(v).numpy() for v in base_model.trainable_variables])
trainable_params_tl = sum([tf.size(v).numpy() for v in transfer_model.trainable_variables])
print(f"\nFrozen parameters (backbone): {frozen_params:,}")
print(f"Trainable parameters (head):  {trainable_params_tl:,}")

In [ ]:
# ─── Train Transfer Learning Model ───
print("Training Transfer Learning Model (frozen backbone)...")
print("=" * 60)

early_stop_tl = callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
)

start_time = time.time()
transfer_history = transfer_model.fit(
    train_ds_tl,
    epochs=10,
    validation_data=val_ds_tl,
    callbacks=[early_stop_tl],
    verbose=1
)
transfer_train_time = time.time() - start_time

print(f"\n{'=' * 60}")
print(f"Training completed in {transfer_train_time:.1f} seconds")
print(f"Epochs trained: {len(transfer_history.history['loss'])}")
print(f"{'=' * 60}")

In [ ]:
# ─── Evaluate Transfer Learning Model ───
tl_train_acc = transfer_history.history['accuracy']
tl_val_acc = transfer_history.history['val_accuracy']
tl_train_loss = transfer_history.history['loss']
tl_val_loss = transfer_history.history['val_loss']

tl_best_val_acc = max(tl_val_acc)
tl_best_epoch = np.argmax(tl_val_acc) + 1

tl_test_loss, tl_test_acc = transfer_model.evaluate(test_ds_tl_prep, verbose=0)

print("=" * 60)
print("TRANSFER LEARNING RESULTS")
print("=" * 60)
print(f"Training time:            {transfer_train_time:.1f}s")
print(f"Epochs trained:           {len(tl_train_acc)}")
print(f"Best validation accuracy: {tl_best_val_acc:.4f} (epoch {tl_best_epoch})")
print(f"Test accuracy:            {tl_test_acc:.4f}")
print(f"Test loss:                {tl_test_loss:.4f}")
print("=" * 60)

# ─── Plot Training Curves ───
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Experiment 3: Transfer Learning (MobileNetV2) — Training Curves', fontsize=14, fontweight='bold')

epochs_range = range(1, len(tl_train_acc) + 1)
ax1.plot(epochs_range, tl_train_acc, 'b-o', label='Training Accuracy', markersize=4)
ax1.plot(epochs_range, tl_val_acc, 'r-o', label='Validation Accuracy', markersize=4)
ax1.axhline(y=tl_test_acc, color='green', linestyle='--', alpha=0.7, label=f'Test Acc ({tl_test_acc:.4f})')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy'); ax1.set_title('Accuracy'); ax1.legend(); ax1.set_ylim([0.4, 1.0])

ax2.plot(epochs_range, tl_train_loss, 'b-o', label='Training Loss', markersize=4)
ax2.plot(epochs_range, tl_val_loss, 'r-o', label='Validation Loss', markersize=4)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss'); ax2.set_title('Loss'); ax2.legend()

plt.tight_layout()
plt.show()

---
## Experiment Comparison — All Three Approaches

In [ ]:
# ─── Experiment Comparison Table ───
print("=" * 80)
print("EXPERIMENT COMPARISON TABLE")
print("=" * 80)

header = f"{'Model':<25s} {'Aug':>5s} {'Transfer':>8s} {'Frozen':>6s} {'Time(s)':>8s} {'Best Val':>9s} {'Test Acc':>9s}"
print(header)
print("-" * 80)

results = {
    'CNN From Scratch':    (baseline_best_val_acc, baseline_test_acc, baseline_train_time),
    'CNN + Augmentation':  (aug_best_val_acc, aug_test_acc, augmented_train_time),
    'Transfer Learning':   (tl_best_val_acc, tl_test_acc, transfer_train_time),
}

rows = [
    ("CNN From Scratch",    "No",   "No",  "N/A"),
    ("CNN + Augmentation",  "Yes",  "No",  "N/A"),
    ("Transfer Learning",   "Appr.","Yes", "Frozen"),
]

for name, aug, tl, frozen in rows:
    val, test, t = results[name]
    print(f"{name:<25s} {aug:>5s} {tl:>8s} {frozen:>6s} {t:>8.1f} {val:>9.4f} {test:>9.4f}")

print("=" * 80)

best_val_model = max(results, key=lambda k: results[k][0])
best_test_model = max(results, key=lambda k: results[k][1])
fastest_model = min(results, key=lambda k: results[k][2])

print(f"\nBest Validation Accuracy: {best_val_model} ({results[best_val_model][0]:.4f})")
print(f"Best Test Accuracy:       {best_test_model} ({results[best_test_model][1]:.4f})")
print(f"Fastest Training:         {fastest_model} ({results[fastest_model][2]:.1f}s)")

In [ ]:
# ─── Visual Comparison ───
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Validation Curves: All Three Experiments', fontsize=14, fontweight='bold')

models_data = [
    ('CNN From Scratch', baseline_val_acc, baseline_val_loss, 'blue'),
    ('CNN + Augmentation', aug_val_acc, aug_val_loss, 'red'),
    ('Transfer Learning', tl_val_acc, tl_val_loss, 'green'),
]

for ax, (name, val_a, val_l, color) in zip(axes, models_data):
    epochs = range(1, len(val_a) + 1)
    ax.plot(epochs, val_a, f'{color}-o', label='Val Accuracy', markersize=4)
    ax.plot(epochs, val_l, f'{color}--s', label='Val Loss', markersize=4, alpha=0.7)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Value'); ax.set_title(name); ax.legend(); ax.set_ylim([0, 1.0])

plt.tight_layout()
plt.show()

# ─── Bar Chart ───
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Final Metrics Comparison', fontsize=14, fontweight='bold')

model_names = ['CNN\nScratch', 'CNN +\nAug', 'Transfer\nLearning']
val_accs = [results[n][0] for n in results]
test_accs = [results[n][1] for n in results]
times = [results[n][2] for n in results]
colors = ['steelblue', 'indianred', 'forestgreen']

x_pos = np.arange(len(model_names))
w = 0.35
b1 = axes[0].bar(x_pos - w/2, val_accs, w, label='Best Val Accuracy', color=colors, alpha=0.7)
b2 = axes[0].bar(x_pos + w/2, test_accs, w, label='Test Accuracy', color=colors, alpha=0.4, edgecolor='black')
axes[0].set_ylabel('Accuracy'); axes[0].set_title('Accuracy Comparison')
axes[0].set_xticks(x_pos); axes[0].set_xticklabels(model_names); axes[0].legend(); axes[0].set_ylim([0.4, 1.0])

for bar in b1:
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01, f'{bar.get_height():.3f}', ha='center', fontsize=9)
for bar in b2:
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01, f'{bar.get_height():.3f}', ha='center', fontsize=9)

b3 = axes[1].bar(x_pos, times, color=colors, alpha=0.8, edgecolor='black')
axes[1].set_ylabel('Training Time (s)'); axes[1].set_title('Training Time Comparison')
axes[1].set_xticks(x_pos); axes[1].set_xticklabels(model_names)
for bar in b3:
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1, f'{bar.get_height():.1f}s', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

---
## Classification Report & Confusion Matrix — Best Model

In [ ]:
# ─── Get Predictions from Best Model ───
if best_test_model == 'Transfer Learning':
    y_pred_probs = transfer_model.predict(test_ds_tl_prep, verbose=0)
elif best_test_model == 'CNN + Augmentation':
    y_pred_probs = augmented_model.predict(test_ds_cnn, verbose=0)
else:
    y_pred_probs = baseline_model.predict(test_ds_norm, verbose=0)

y_pred = (y_pred_probs > 0.5).astype(int).flatten()

# Get true labels
y_true = np.concatenate([labels.numpy() for _, labels in test_ds_cnn], axis=0).astype(int)

print(f"=" * 60)
print(f"CLASSIFICATION REPORT — {best_test_model}")
print(f"=" * 60)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# ─── Confusion Matrix ───
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)
ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
       ylabel='True Label', xlabel='Predicted Label',
       title=f'Confusion Matrix — {best_test_model}')

thresh = cm.max() / 2.0
for i in range(2):
    for j in range(2):
        ax.text(j, i, format(cm[i, j], 'd'), ha='center', va='center',
                color='white' if cm[i, j] > thresh else 'black', fontsize=16)

plt.tight_layout()
plt.show()

---
## Final Day 2 Findings

### 1. How did the CNN from scratch perform?

The baseline CNN (3 conv blocks with max-pooling) achieved a test accuracy of **{:.4f}** after **{:.1f}** seconds of training. The architecture successfully learned visual features (edges, textures, patterns) from the skin lesion images without any external pre-training.

### 2. Did data augmentation improve generalization?

Augmentation {'improved' if aug_test_acc > baseline_test_acc else 'did not improve'} test accuracy (from **{:.4f}** to **{:.4f}**). The augmentation pipeline applied random horizontal flips, rotations (±15%), and zooms (±15%) during training only.

### 3. Did augmentation reduce signs of overfitting?

The train-validation accuracy gap was **{:.4f}** for the baseline and **{:.4f}** for the augmented model. {'Augmentation reduced the overfitting gap.' if aug_gap < gap else 'The gap was comparable.'}

### 4. How did transfer learning compare with training from scratch?

Transfer learning using a frozen MobileNetV2 backbone achieved a test accuracy of **{:.4f}**, {'significantly outperforming' if tl_test_acc > baseline_test_acc + 0.02 else 'comparable to'} the from-scratch CNN (**{:.4f}**). The pre-trained features from ImageNet generalized well to skin lesion classification.

### 5. Which approach achieved the best validation/test performance?

**{best_test_model}** achieved the highest test accuracy at **{:.4f}**.

### 6. Which approach trained fastest?

**{fastest_model}** completed training in **{:.1f}** seconds. Transfer learning is fast per epoch (frozen backbone), but processes larger 224×224 images.

### 7. Why did the best approach perform better?

The best approach benefited from {'pre-learned visual features that transferred well from ImageNet to skin lesion images' if best_test_model == 'Transfer Learning' else 'a well-matched architecture and training procedure'}. Transfer learning is particularly effective because low-level visual features (edges, textures, color patterns) are universal across image domains.

### 8. What trade-offs exist between the three approaches?

| Trade-off | CNN From Scratch | CNN + Augmentation | Transfer Learning |
|:----------|:-----------------|:-------------------|:------------------|
| **Training speed** | Fast (small images) | Slightly slower | Fast (frozen backbone) |
| **Overfitting risk** | Higher | Lower | Lower |
| **Data efficiency** | Needs more data | Better with less data | Best with limited data |
| **Architecture control** | Full | Full | Constrained |

### 9. What was learned from the experiment?

1. **Convolution + Pooling** is the foundation of image classification
2. **Data augmentation** is a simple, effective regularization technique
3. **Transfer learning** provides strong results even with frozen weights
4. **Architecture must match data** — these CNN techniques do NOT apply to our project's tabular Heart Disease data

### 10. How does this connect to the next stages of Week 7?

Day 3 introduces **RNN/LSTM** for sequential data. The principle remains: **choose the architecture that matches your data's structure.**

In [ ]:
# ─── Auto-Generated Findings ───
print("=" * 80)
print("FINAL DAY 2 FINDINGS — Based on Actual Experimental Results")
print("=" * 80)
print(f"""
1. CNN FROM SCRATCH
   Test Accuracy: {baseline_test_acc:.4f}
   Training Time: {baseline_train_time:.1f}s
   The baseline CNN with 3 conv blocks + max pooling learned effective visual features.

2. DATA AUGMENTATION
   Test Accuracy: {aug_test_acc:.4f} ({'+' if aug_test_acc >= baseline_test_acc else ''}{(aug_test_acc - baseline_test_acc)*100:.2f}% vs baseline)
   Training Time: {augmented_train_time:.1f}s
   Overfitting gap: Baseline={gap:.4f}, Augmented={aug_gap:.4f}

3. TRANSFER LEARNING (MobileNetV2, frozen)
   Test Accuracy: {tl_test_acc:.4f} ({'+' if tl_test_acc >= baseline_test_acc else ''}{(tl_test_acc - baseline_test_acc)*100:.2f}% vs baseline)
   Training Time: {transfer_train_time:.1f}s

4. BEST APPROACH: {best_test_model}
   Test Accuracy: {results[best_test_model][1]:.4f}

5. FASTEST TRAINING: {fastest_model}
   Time: {results[fastest_model][2]:.1f}s
""")
print("=" * 80)

---
## Connection to Future Week 7 Work

### Week 7 Curriculum Progression

```
Weeks 1–5: Classical ML foundations
Week 6:    Dense Neural Networks (our core project model)
Day 1 ✅:  Convolution fundamentals — hand-defined filters, feature maps, parameter sharing
Day 2 ✅:  CNN + Pooling + Data Augmentation + Transfer Learning (this notebook)
Day 3:     RNN / LSTM — sequential data architectures
Day 4:     Attention / Transformers — modern sequence modeling
Day 5:     Core Model Advancement + Sprint Review
```

Days 1–2 explored **image** architectures (CNNs). Days 3–4 will explore **sequential** architectures (RNNs, Transformers). The core project continues to use the **dense network from Week 6** enhanced by Sprint 2 improvements.

---
## Final Review Checklist

### Architecture Decision
- [x] Project data type identified as TABULAR
- [x] CNN documented as educational experiment using real medical images
- [x] Architecture-data matching principle explained

### Experiment 1: CNN From Scratch
- [x] CNN with Conv2D, MaxPooling2D, Flatten, Dense implemented
- [x] Trained and evaluated on real image data
- [x] Training/validation curves generated
- [x] Training time and accuracy recorded

### Experiment 2: CNN + Data Augmentation
- [x] RandomFlip, RandomRotation, RandomZoom applied
- [x] Augmentation only during training (no leakage)
- [x] Trained and evaluated; comparison with baseline

### Experiment 3: Transfer Learning
- [x] MobileNetV2 pre-trained on ImageNet loaded
- [x] Backbone frozen; new classification head added
- [x] Trained and evaluated

### Comparison & Documentation
- [x] Fair comparison table with all three approaches
- [x] Visual comparison plots and bar charts
- [x] Classification report and confusion matrix
- [x] Best approach selected based on actual evidence
- [x] 10-question findings section
- [x] Connection to future Week 7 work

### Quality
- [x] No fabricated metrics
- [x] No augmentation leakage
- [x] Reproducible (random seeds set)
- [x] No unnecessary dependencies